# Optimización de Umbrales (Threshold Tuning)

Por defecto, usamos un umbral de 0.5 para decidir si un tiro es gol o no. Sin embargo, en datos desbalanceados, bajar este umbral puede ayudarnos a capturar más goles reales.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
import matplotlib.pyplot as plt

# Cargar datos y entrenar el mismo modelo lineal
df = pd.read_csv('../data/processed/shots_features.csv')
features = ['distance', 'angle', 'is_header', 'is_big_chance', 'is_penalty', 'is_counter']
df = df.dropna(subset=features + ['is_goal'])
X = df[features]
y = df['is_goal'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_probs = model.predict(X_test)

### Comparación de Umbrales

In [ ]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
results = []

for t in thresholds:
    y_pred = [1 if p > t else 0 for p in y_probs]
    results.append({
        'Umbral': t,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall (Gol)': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

### Visualización del Trade-off
A medida que bajamos el umbral, el Recall sube (encontramos más goles) pero la Precisión baja (cometemos más errores).

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df_results['Umbral'], df_results['Precision'], label='Precision', marker='o')
plt.plot(df_results['Umbral'], df_results['Recall (Gol)'], label='Recall', marker='o')
plt.plot(df_results['Umbral'], df_results['F1-Score'], label='F1-Score (Balance)', marker='o', linestyle='--')
plt.axvline(0.2, color='red', linestyle=':', label='Umbral Recomendado (0.2)')
plt.title('Impacto del umbral en las métricas de xG')
plt.xlabel('Umbral de Probabilidad')
plt.ylabel('Puntaje')
plt.legend()
plt.grid(True)
plt.show()